# MLflow Starter: AG News Classification



# 1. Inside the repo

This project contains:

- `data/train.csv`
- `data/test.csv`
- `data/test_labels.csv`
- `mlflow_starter.ipynb`
- `requirements.txt`

Run the notebook from the repository root so the relative data paths resolve correctly.

# 2. What is a virtual environment?

A virtual environment is an isolated Python installation for one project. It keeps package versions separate from other projects and makes setup reproducible.

# 3. Create and activate it

**Windows PowerShell**

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
```

**Windows Command Prompt**

```bat
python -m venv .venv
.venv\Scripts\activate
```

**macOS/Linux**

```bash
python3 -m venv .venv
source .venv/bin/activate
```

# 4. Install from `requirements.txt`

```bash
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
```

Run this from the repository root after activating `.venv`.

# 5. Start the MLflow tracking server

From the repository root, run this in a separate terminal and leave it running:

```bash
mlflow server --host 0.0.0.0 --port 5000
```

The local tracking UI is available at `http://localhost:5000`.

# 7. A tour of the MLflow UI

Open the tracking URL and inspect:

- the experiment list
- run parameters and metrics
- artifacts and model signatures
- registered model versions

Refresh the experiment after each training run.

# 8. Open `mlflow_starter.ipynb`

Keep the MLflow tracking server running in the background while you execute the notebook sections.

### Imports

In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

### Connect to MLflow and set the experiment

In [2]:
TRACKING_URI = "http://localhost:5000"


In [3]:
mlflow.set_tracking_uri(TRACKING_URI)


In [4]:
mlflow.set_experiment("News_classification")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

2026/09/05 20:15:48 INFO mlflow.tracking.fluent: Experiment with name 'News_classification' does not exist. Creating a new experiment.


Tracking URI: http://localhost:5000


### Load the AG News data

The test labels are stored separately, so they are loaded and aligned with `test.csv` by row order.

In [5]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")
test_labels_df = pd.read_csv("data/test_labels.csv", index_col=0)

X_train = train_df["Description"]
y_train = train_df["Class Index"]

TEST_SAMPLE_SIZE = 5000
test_positions = test_df.sample(
    n=min(TEST_SAMPLE_SIZE, len(test_df), len(test_labels_df)),
    random_state=42
).index.to_numpy()

X_test = test_df.iloc[test_positions]["Description"].fillna("").reset_index(drop=True)
y_test = test_labels_df.iloc[test_positions]["Class Index"].reset_index(drop=True)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
X_train.head()

Training rows: 96,000
Test rows: 5,000


0    Please donate now to our Fall Fund Drive to he...
1    AP - Maybe the Atlanta Falcons aren't a one-ma...
2    KARACHI, October 28 (Online): Bruised and batt...
3    Two villages in Carmarthenshire are the first ...
4    A major re-ordering of South Africa #39;s gold...
Name: Description, dtype: str

### Manually log the BoW baseline

This section trains the Bag-of-Words plus Logistic Regression baseline and manually records the run in MLflow. The commands used are:

- `mlflow.start_run(run_name="BoW_LogReg_Manual")`: opens an MLflow run and groups all logged information under one run name. The `with` block automatically ends the run.

- `mlflow.log_params({...})`: records configuration values such as the vectorizer, vocabulary limit, classifier, and training settings.

- `mlflow.log_metrics({...})`: records numeric evaluation results, including accuracy and weighted F1 score.

- `infer_signature(sample_input, bow_logreg.predict(sample_input))`: infers the model input and output schema from example text and predictions.

- `mlflow.sklearn.log_model(...)`: saves the trained scikit-learn pipeline as an MLflow model. The `name` argument gives the model artifact a name, and `signature` documents the expected input and output format.

The model itself is trained with `fit`, predictions are generated with `predict`, and the resulting metrics and model artifact can be inspected in the MLflow UI.

In [6]:
mlflow.sklearn.autolog(disable=True)
mlflow.end_run()

bow_logreg = Pipeline([
    ("vectorizer", CountVectorizer(max_features=2000)),
    ("classifier", LogisticRegression(max_iter=500))
])

with mlflow.start_run(run_name="BoW_LogReg_Manual"):
    bow_logreg.fit(X_train, y_train)
    bow_predictions = bow_logreg.predict(X_test)
    bow_accuracy = accuracy_score(y_test, bow_predictions)
    bow_f1 = f1_score(y_test, bow_predictions, average="weighted")

    mlflow.log_params({
        "vectorizer": "CountVectorizer",
        "vectorizer_max_features": 2000,
        "classifier": "LogisticRegression",
        "classifier_max_iter": 500
    })
    mlflow.log_metrics({
        "accuracy": float(bow_accuracy),
        "f1_score": float(bow_f1)
    })

    sample_input = X_test.head(2).tolist()
    signature = infer_signature(sample_input, bow_logreg.predict(sample_input))
    mlflow.sklearn.log_model(
        sk_model=bow_logreg,
        name="bagofwords_logreg",
        signature=signature
    )

print(f"Accuracy: {bow_accuracy:.4f}")
print(f"F1 score: {bow_f1:.4f}")

🏃 View run BoW_LogReg_Manual at: http://localhost:5000/#/experiments/1/runs/78bc598ef3b746168efeb5ecdef0632d
🧪 View experiment at: http://localhost:5000/#/experiments/1
Accuracy: 0.8658
F1 score: 0.8653


### Hyperparameter tuning with MLflow autologging

This section evaluates every hyperparameter combination as a separate MLflow run. Each run name includes the model and parameter values, while MLflow autologging records the fitted model, parameters, training metrics, and artifacts. The held-out test sample is used for the final accuracy and weighted F1 comparison.

In [7]:
# mlflow.sklearn.autolog(log_models=True)

tuning_configs = {
    "TFIDF_DecisionTree": (
        Pipeline([
            ("vectorizer", TfidfVectorizer(max_features=2000)),
            ("classifier", DecisionTreeClassifier(random_state=42))
        ]),
        {
            "classifier__max_depth": [5, 10, 20],
            "classifier__min_samples_split": [2, 5]
        }
    ),
    "TFIDF_RandomForest": (
        Pipeline([
            ("vectorizer", TfidfVectorizer(max_features=2000)),
            ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1))
        ]),
        {
            "classifier__n_estimators": [50, 100],
            "classifier__max_depth": [10, 20],
            "classifier__min_samples_split": [2, 5]
        }
    )
}

tuning_results = {}
for model_name, (base_pipeline, parameter_grid) in tuning_configs.items():
    tuning_results[model_name] = []

    for parameters in ParameterGrid(parameter_grid):
        parameter_text = "__".join(
            f"{key.split('__')[-1]}-{value}"
            for key, value in parameters.items()
        )
        run_name = f"{model_name}__{parameter_text}"
        pipeline = base_pipeline.set_params(**parameters)

        with mlflow.start_run(run_name=run_name):
            pipeline.fit(X_train, y_train)
            predictions = pipeline.predict(X_test)
            accuracy = accuracy_score(y_test, predictions)
            weighted_f1 = f1_score(y_test, predictions, average="weighted")
            mlflow.log_metrics({
                "test_accuracy": float(accuracy),
                "test_f1_score": float(weighted_f1)
            })

            sample_input = X_test.head(2).tolist()
            signature = infer_signature(sample_input, pipeline.predict(sample_input))
            mlflow.sklearn.log_model(
                sk_model=pipeline,
                name=model_name,
                signature=signature
            )

            tuning_results[model_name].append({
                "run_name": run_name,
                "parameters": parameters,
                "accuracy": accuracy,
                "f1_score": weighted_f1
            })
            print(
                f"{run_name}: "
                f"accuracy={accuracy:.4f}, "
                f"f1={weighted_f1:.4f}"
            )

print("Refresh the MLflow UI to compare every parameterized tuning run.")

TFIDF_DecisionTree__max_depth-5__min_samples_split-2: accuracy=0.3804, f1=0.3425
🏃 View run TFIDF_DecisionTree__max_depth-5__min_samples_split-2 at: http://localhost:5000/#/experiments/1/runs/37e0b9e66f6648bd8681a11cc167f6d9
🧪 View experiment at: http://localhost:5000/#/experiments/1
TFIDF_DecisionTree__max_depth-5__min_samples_split-5: accuracy=0.3804, f1=0.3425
🏃 View run TFIDF_DecisionTree__max_depth-5__min_samples_split-5 at: http://localhost:5000/#/experiments/1/runs/56eada40338944e98785352893bc6ad8
🧪 View experiment at: http://localhost:5000/#/experiments/1
TFIDF_DecisionTree__max_depth-10__min_samples_split-2: accuracy=0.4676, f1=0.4539
🏃 View run TFIDF_DecisionTree__max_depth-10__min_samples_split-2 at: http://localhost:5000/#/experiments/1/runs/56a97309643f44ebb7c746ec59f0dc37
🧪 View experiment at: http://localhost:5000/#/experiments/1
TFIDF_DecisionTree__max_depth-10__min_samples_split-5: accuracy=0.4674, f1=0.4536
🏃 View run TFIDF_DecisionTree__max_depth-10__min_samples_spli

### 16. Compare runs and select the best model

After the tuning section finishes, open the MLflow UI at `http://localhost:5000` and open the `News_classification` experiment.

1. Compare the runs using the `test_f1_score` and `test_accuracy` metrics.
2. Open the run with the strongest evaluation result. The run name shows its model and hyperparameters.
3. Copy the complete **Run ID** from that run's details page. Do not copy the run name.
4. Paste the Run ID into the prediction section below and run it.

Each tuning run stores its model under a descriptive artifact path: `TFIDF_DecisionTree` or `TFIDF_RandomForest`. Use the path matching the model family of the selected run:

`runs:/RUN_ID/TFIDF_DecisionTree`

or

`runs:/RUN_ID/TFIDF_RandomForest`

AG News class mapping:

| Class Index | Category |
| --- | --- |
| 1 | World |
| 2 | Sports |
| 3 | Business |
| 4 | Sci/Tech |

In [ ]:
model_uri = "runs:/<run ID>/<Model name>"
pro_model = mlflow.sklearn.load_model(model_uri)

sample_text = ["The stock market crashed today as tech shares plummeted."]
prediction = pro_model.predict(sample_text)

print(f"Loaded model: {model_uri}")
print(f"Prediction for sample text: Class {int(prediction[0])}")

Loaded model: runs:/37e0b9e66f6648bd8681a11cc167f6d9/TFIDF_DecisionTree
Prediction for sample text: Class 4


# 6. Reopen existing MLflow runs

To see runs that were already logged, start MLflow with the same backend database and artifact directory that were used when the runs were created. Run this from the repository root:

**Windows PowerShell or Command Prompt**

```bat
mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlartifacts --host 0.0.0.0 --port 5000
```

- `mlflow.db` contains the experiment, run, metric, parameter, and model metadata.
- `mlartifacts` contains the logged model files and other artifacts referenced by those runs.
- If the database has another filename or is in another folder, replace `sqlite:///mlflow.db` with its correct path. Keep the artifact directory unchanged.

Then open `http://localhost:5000`, select the `News_classification` experiment.

## 19. Task: DistilBERT on AG News, tracked in MLflow

Extend this notebook by fine-tuning DistilBERT for the four AG News classes. Track the training run in MLflow, including hyperparameters, evaluation metrics, tokenizer/model artifacts, and an input-output signature. Compare the transformer model with the classical baselines above.